# Source
https://vkvideo.ru/video-145052891_456248475

In [1]:
import warnings

In [2]:
warnings.simplefilter(action="ignore")

In [3]:
import pickle
import json
import random

import numpy as np

from lightfm import LightFM
from lightfm.data import Dataset

from replay.metrics import MAP, MRR, RocAuc, HitRate, NDCG, Coverage, Experiment
from replay.splitters import RandomSplitter

In [4]:
SEED = 23

In [5]:
random.seed(SEED)
np.random.seed(SEED)

In [6]:
USER_COL = "user_name"
ITEM_COL = "item_title"
RATING_COL = "rating"

In [7]:
MODEL = "google/gemini-2.5-flash"

In [8]:
output_dir_name = MODEL.split("/")[1]

In [9]:
with open(f"{output_dir_name}/artificial_profiles_scores.pkl", "rb") as f:
    interactions = pickle.load(f)

with open(f"{output_dir_name}/artificial_profiles.json", "r", encoding="utf-8") as f:
    features_by_user = json.load(f)

with open(f"{output_dir_name}/titles_with_tags_dict_processed.pkl", "rb") as f:
    features_by_item = pickle.load(f)

print("Data loaded successfully:")
print(f"  - Users: {len(interactions)}")
print(f"  - Items: {len(features_by_item)}")
print(
    f"  - Total interactions: {sum(len(ratings) for ratings in interactions.values())}"
)

Data loaded successfully:
  - Users: 31
  - Items: 1147
  - Total interactions: 2697


In [10]:
sample_user = list(features_by_user.keys())[0]
print(f"Sample user: {sample_user}")
print(f"Bio: {features_by_user[sample_user]['bio'][:200]}...")
print(f"Tags: {features_by_user[sample_user]['tags']}")

Sample user: cultural_studies_enthusiast
Bio: A user interested in various aspects of culture, history, and social phenomena, with a particular focus on non-Western cultures, literature, and media....
Tags: ['arts_and_culture', 'middle_eastern_studies', 'asian_studies', 'folklore', 'postcolonialism']


In [11]:
user_to_test = random.choice(list(features_by_user.keys()))

In [12]:
user_to_test

'urban_cultural_researcher'

In [13]:
sample_item = list(features_by_item.keys())[0]
print(f"Sample item: {sample_item[:80]}...")
print(f"Tags: {features_by_item[sample_item]}")

Sample item: Исследование приоритетов и механизмов реализации отраслевых (секторальных) полит...
Tags: ['BRICS', 'development_studies', 'economics', 'politics', 'russian_studies']


In [14]:
import numpy as np
import pandas as pd

In [15]:
data = []

for user in interactions:
    for payload in interactions[user]:
        if interactions[user][payload] > 3:
            data.append(
                (user, json.loads(payload).get("title"), interactions[user][payload])
            )

df = pd.DataFrame(data=data, columns=["user_name", "item_title", "rating"])

In [16]:
df.shape

(2305, 3)

In [17]:
df.head()

,user_name,item_title,rating
0,cultural_studies_enthusiast,Запуск регулярного подкаста НИУ ВШЭ о странах ...,5
1,cultural_studies_enthusiast,История турецкой литературы и фольклора,5
2,cultural_studies_enthusiast,Китайский клуб НИУ ВШЭ 2025/2026,4
3,cultural_studies_enthusiast,Работа с материалами фольклорных экспедиций 2025,5
4,cultural_studies_enthusiast,Современное Искусство Ближнего Востока,5


In [18]:
data = []

for user in features_by_user:
    data.append((user, features_by_user[user]["tags"]))

df_users = pd.DataFrame(data=data, columns=["user_name", "features"])

In [19]:
df_users.head()

,user_name,features
0,cultural_studies_enthusiast,"[arts_and_culture, middle_eastern_studies, asi..."
1,educational_content_developer,"[education, teaching_methods, e-learning, cont..."
2,media_content_creator,"[content_marketing, smm, digital_media, promot..."
3,geopolitical_analyst_brics_middle_east,"[brics, global_politics, middle_eastern_studie..."
4,cultural_exchange_coordinator,"[cultural_exchange, arts_and_culture, event_ma..."


In [20]:
data = []

for item in features_by_item:
    data.append((item, features_by_item[item]))

df_items = pd.DataFrame(data=data, columns=["item_title", "features"])

In [21]:
df_items.head()

,item_title,features
0,Исследование приоритетов и механизмов реализац...,"[BRICS, development_studies, economics, politi..."
1,Антрополе - научно-популярный видео-подкаст о ...,"[digital_media, humanities, podcasting, sociol..."
2,"Разработка, создание и ведение сайта, посвящен...","[arts, culture, data_science, digital_media, h..."
3,Перевод с английского языка коллективной моног...,"[criminology, humanities, translation]"
4,Сеть военно-политических союзов в Евразии: баз...,"[data_science, eurasian_studies, politics, rus..."


In [22]:
dfg = df.groupby(by=ITEM_COL).agg({USER_COL: "nunique"}).reset_index()
dfg.columns = ["item_title", "user_name_count"]
dfg.sort_values(by="user_name_count", ascending=False)

,item_title,user_name_count
16,Digital Days на Факультете гуманитарных наук: ...,10
800,Цифровизация и отдача от образования в России,10
250,Китайский клуб НИУ ВШЭ 2025/2026,9
48,"«Тройка» Северо-Восточной Азии: состояние, про...",8
151,Волонтёры: Олимпиадный марафон,8
...,...,...
111,Ведение официального сайта ридинг-семинара «Ме...,1
684,Создание и реализация бизнес-инкубатора корпор...,1
685,Создание и редактирование контента для студенч...,1
428,Подбор и анализ телеграм-блогеров для выстраив...,1


In [23]:
df[USER_COL].nunique()

31

In [24]:
df[ITEM_COL].nunique()

850

In [25]:
train_dfs = []
test_dfs = []

for user_name in sorted(df.user_name.unique()):
    user_train, user_test = RandomSplitter(
        query_column=USER_COL,
        item_column=ITEM_COL,
        test_size=0.2,
    ).split(df[df.user_name == user_name])
    train_dfs.append(user_train)
    test_dfs.append(user_test)

train = pd.concat(train_dfs)
test = pd.concat(test_dfs)

In [26]:
df.shape

(2305, 3)

In [27]:
train.shape

(1841, 3)

In [28]:
train.groupby(by=["user_name"]).agg({"item_title": "nunique"}).reset_index()

,user_name,item_title
0,african_and_middle_eastern_studies_researcher,12
1,ai_in_education_and_law_enthusiast,150
2,ai_in_education_innovator,62
3,art_and_culture_enthusiast,16
4,business_strategist,60
5,cultural_cinema_enthusiast,18
6,cultural_exchange_coordinator,35
7,cultural_studies_enthusiast,22
8,data_management_specialist,23
9,east_asia_cultural_specialist,22


In [29]:
train.user_name.nunique()

31

In [30]:
test.shape

(464, 3)

In [31]:
test.user_name.nunique()

31

In [32]:
test.groupby(by=["user_name"]).agg({"item_title": "nunique"}).reset_index()

,user_name,item_title
0,african_and_middle_eastern_studies_researcher,3
1,ai_in_education_and_law_enthusiast,38
2,ai_in_education_innovator,16
3,art_and_culture_enthusiast,4
4,business_strategist,15
5,cultural_cinema_enthusiast,5
6,cultural_exchange_coordinator,9
7,cultural_studies_enthusiast,5
8,data_management_specialist,6
9,east_asia_cultural_specialist,6


In [33]:
ALL_USERS = train[USER_COL].unique().tolist()
ALL_ITEMS = train[ITEM_COL].unique().tolist()

users = df_users[df_users[USER_COL].isin(ALL_USERS)]
items = df_items[df_items[ITEM_COL].isin(ALL_ITEMS)]

In [34]:
dataset = Dataset()

In [35]:
dataset.fit_partial(ALL_USERS, ALL_ITEMS)

In [36]:
user_mapping = dataset.mapping()[0]
item_mapping = dataset.mapping()[2]

inv_user_mapping = {v: k for k, v in user_mapping.items()}
inv_item_mapping = {v: k for k, v in item_mapping.items()}

In [37]:
train_interactions, train_weights = dataset.build_interactions(
    train[[USER_COL, ITEM_COL, RATING_COL]].values
)

In [38]:
train_interactions

<COOrdinate sparse matrix of dtype 'int32'
	with 1841 stored elements and shape (31, 786)>

In [39]:
train_weights

<COOrdinate sparse matrix of dtype 'float32'
	with 1841 stored elements and shape (31, 786)>

In [40]:
class LightFM4Rec:
    def __init__(
        self,
        model,
        user_mapping,
        item_mapping,
        inv_user_mapping,
        inv_item_mapping,
        user_col,
        item_col,
        rating_col,
    ):
        self.model = model
        self.user_mapping = user_mapping
        self.item_mapping = item_mapping
        self.inv_user_mapping = inv_user_mapping
        self.inv_item_mapping = inv_item_mapping
        self.user_col = user_col
        self.item_col = item_col
        self.rating_col = rating_col

    def fit(
        self,
        rating_matrix,
        train_interactions,
        user_features=None,
        item_features=None,
        epochs=16,
    ):
        self.user_features = user_features
        self.item_features = item_features
        self.train_interactions = train_interactions
        self.model.fit(
            rating_matrix,
            user_features=self.user_features,
            item_features=self.item_features,
            epochs=epochs,
        )

    def _get_lfm_pred(self, user_id):
        pred = self.model.predict(
            user_ids=user_id,
            item_ids=self.item_ids,
            user_features=self.user_features,
            item_features=self.item_features,
        )
        return pred

    def predict(self, test, interaction_matrix, user_col, filter_seen=True, k=10):
        user_ids = test[self.user_col].map(self.user_mapping).unique()
        self.item_ids = list(self.item_mapping.values())

        pred = pd.DataFrame(user_ids, columns=[user_col])
        scores = np.vstack(pred[user_col].apply(self._get_lfm_pred).values)

        if filter_seen:
            scores += np.nan_to_num(interaction_matrix.todense()[user_ids] * -np.inf)

        ind_part = np.argpartition(scores, -k)[:, -k:].copy()
        scores_not_sorted = np.take_along_axis(scores, ind_part, axis=1)
        ind_sorted = np.argsort(scores_not_sorted, axis=1)
        scores_sorted = np.sort(scores_not_sorted, axis=1)
        indices = np.take_along_axis(ind_part, ind_sorted, axis=1)

        preds = pd.DataFrame(
            {
                self.user_col: user_ids,
                self.item_col: np.flip(indices, axis=1).tolist(),
                self.rating_col: np.flip(scores_sorted, axis=1).tolist(),
            }
        ).explode([self.item_col, self.rating_col])
        preds[self.user_col] = preds[self.user_col].map(self.inv_user_mapping)
        preds[self.item_col] = preds[self.item_col].map(self.inv_item_mapping)

        return preds

In [41]:
lfm = LightFM(
    random_state=SEED,
    loss="logistic",
    no_components=16,
)

In [42]:
model = LightFM4Rec(
    model=lfm,
    user_mapping=user_mapping,
    item_mapping=item_mapping,
    inv_user_mapping=inv_user_mapping,
    inv_item_mapping=inv_item_mapping,
    user_col=USER_COL,
    item_col=ITEM_COL,
    rating_col=RATING_COL,
)

In [43]:
model.fit(train_weights, train_interactions)

In [44]:
preds_wo_features = model.predict(test, train_interactions, USER_COL)

In [45]:
preds_wo_features[preds_wo_features["user_name"] == user_to_test]

,user_name,item_title,rating
29,urban_cultural_researcher,Цифровизация и отдача от образования в России,1.906663
29,urban_cultural_researcher,Китайский клуб НИУ ВШЭ 2025/2026,1.89295
29,urban_cultural_researcher,Digital Days на Факультете гуманитарных наук: ...,1.890984
29,urban_cultural_researcher,"«Тройка» Северо-Восточной Азии: состояние, про...",1.884981
29,urban_cultural_researcher,Особенности экономического сотрудничества Паки...,1.86811
29,urban_cultural_researcher,ОРЗ Изучении практик многоязычного обучения в ...,1.846256
29,urban_cultural_researcher,Фестиваль языков и культур: подготовка и прове...,1.822039
29,urban_cultural_researcher,Менторы для иностранных слушателей центра русс...,1.81972
29,urban_cultural_researcher,Медиасопровождение Лицейских олимпиадных сборо...,1.818769
29,urban_cultural_researcher,Проект Network for Asia Vision Initiative (NAVI),1.80728


In [46]:
user_tags = users["features"].explode().unique()

In [47]:
item_tags = items["features"].explode().unique()

In [48]:
dataset.fit_partial(user_features=user_tags, item_features=item_tags)

In [49]:
sparse_u_features = dataset.build_user_features(
    [[row.user_name, row.features] for row in users.reset_index().itertuples()]
)

In [50]:
sparse_i_features = dataset.build_item_features(
    [[row.item_title, row.features] for row in items.reset_index().itertuples()]
)

In [51]:
lfm = LightFM(
    random_state=SEED,
    loss="logistic",
    no_components=16,
)

In [52]:
model = LightFM4Rec(
    model=lfm,
    user_mapping=user_mapping,
    item_mapping=item_mapping,
    inv_user_mapping=inv_user_mapping,
    inv_item_mapping=inv_item_mapping,
    user_col=USER_COL,
    item_col=ITEM_COL,
    rating_col=RATING_COL,
)

In [53]:
model.fit(train_weights, train_interactions, sparse_u_features, sparse_i_features)

In [54]:
preds_w_features = model.predict(test, train_interactions, USER_COL)

In [55]:
preds_w_features[preds_w_features["user_name"] == user_to_test]

,user_name,item_title,rating
29,urban_cultural_researcher,Научные Кураторы НИУ ВШЭ: вовлечение иностранн...,3.522624
29,urban_cultural_researcher,Школа будущего международника: десятый сезон (...,3.518834
29,urban_cultural_researcher,Разработка периодического издания в сфере гума...,3.502752
29,urban_cultural_researcher,"ТГ-канал ""Школы юного историка""",3.50254
29,urban_cultural_researcher,Разработка и реализация Школы социальных и эко...,3.494484
29,urban_cultural_researcher,Медиакоманда ЭКСТРА,3.486665
29,urban_cultural_researcher,Межкультурное взаимодействие студентов России ...,3.48453
29,urban_cultural_researcher,Sprachbrücke - Языковой мост,3.464699
29,urban_cultural_researcher,Осенняя Школа юного социолога: ШЮС 2025,3.404202
29,urban_cultural_researcher,Социологическая теория и исследования,3.397139


In [56]:
K = [10]
metrics = Experiment(
    [
        MAP(K),
        MRR(K),
        RocAuc(10),
        NDCG(K),
        HitRate(K),
        Coverage(K),
    ],
    test,
    train,
    query_column=USER_COL,
    item_column=ITEM_COL,
    rating_column=RATING_COL,
)

In [57]:
metrics.add_result("LFM_wo_features", preds_wo_features)
metrics.add_result("LFM_w_features", preds_w_features)

In [58]:
metrics.results

,MAP@10,MRR@10,RocAuc@10,NDCG@10,HitRate@10,Coverage@10
LFM_wo_features,0.017307,0.136329,0.205869,0.043147,0.258065,0.024173
LFM_w_features,0.006617,0.047350,0.049667,0.018830,0.129032,0.027990


In [59]:
data = []

for user_name in sorted(test.user_name.unique()):
    data.append(
        (
            user_name,
            len(
                set(
                    preds_w_features[
                        preds_w_features["user_name"] == user_name
                    ].item_title.unique()
                ).intersection(test[test["user_name"] == user_name].item_title.unique())
            ),
            test[test["user_name"] == user_name].item_title.nunique(),
        )
    )

check = pd.DataFrame(data=data, columns=["user_name", "intersection", "all_test"])
check["share"] = check.apply(lambda x: x.intersection / x.all_test, axis=1)

In [60]:
check.sort_values(by=["share", "intersection"], ascending=[False, False]).reset_index(
    drop=True
)

,user_name,intersection,all_test,share
0,educational_content_developer,3,30,0.100000
1,geopolitical_analyst,1,24,0.041667
2,sociology_and_cultural_studies_enthusiast,1,26,0.038462
3,ai_in_education_and_law_enthusiast,1,38,0.026316
4,african_and_middle_eastern_studies_researcher,0,3,0.000000
5,ai_in_education_innovator,0,16,0.000000
6,art_and_culture_enthusiast,0,4,0.000000
7,business_strategist,0,15,0.000000
8,cultural_cinema_enthusiast,0,5,0.000000
9,cultural_exchange_coordinator,0,9,0.000000


In [61]:
with open(f"{output_dir_name}/lightfm_sber_model.pkl", "wb") as f:
    pickle.dump(model, f)

In [62]:
with open(f"{output_dir_name}/preds_wo_features.pkl", "wb") as f:
    pickle.dump(preds_wo_features, f)

In [63]:
with open(f"{output_dir_name}/preds_w_features.pkl", "wb") as f:
    pickle.dump(preds_w_features, f)